# Pollution Index Scaled - Hourly GeoJSON Map

This notebook follows the same map style as `Visualizse_pollution_index_hourly.ipynb`:

- OpenStreetMap tile background.
- District/province polygons from GeoJSON.
- Green-yellow-red pollution scale from 0 to 100.
- A selected hourly view by `TARGET_DATE` and `TARGET_HOUR`.

The map can also be switched to yearly average mode by changing `MAP_MODE` to `"average"`.

In [ ]:
import json
import re
import unicodedata
from pathlib import Path
from urllib.request import urlopen

import pandas as pd
import plotly.express as px
from IPython.display import display


def u(text):
    return text.encode("ascii").decode("unicode_escape")


# =========================
# CONFIG
# =========================
TARGET_DATE = "2025-01-01"
TARGET_HOUR = "00:00"
MAP_MODE = "hourly"  # "hourly" or "average"
SHOW_PROVINCE_MAPS = True


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "data" / "processed" / "pollution_index_scaled").exists():
            return path
    raise FileNotFoundError("Cannot find data/processed/pollution_index_scaled")


ROOT_DIR = find_project_root()
POLLUTION_DIR = ROOT_DIR / "data" / "processed" / "pollution_index_scaled"

GADM_ADMIN1_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_VNM_1.json"
GADM_ADMIN2_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_VNM_2.json"
GADM_ADMIN1_LOCAL_PATH = ROOT_DIR / "data" / "raw" / "geojson" / "gadm41_VNM_1.geojson"
GADM_ADMIN2_LOCAL_PATH = ROOT_DIR / "data" / "raw" / "geojson" / "gadm41_VNM_2.geojson"

PROVINCES = {
    "ba_ria_vung_tau": {
        "name": u("B\\u00e0 R\\u1ecba - V\\u0169ng T\\u00e0u"),
        "file": "pollution_index_scaled_ba_ria_vung_tau_2025.csv",
        "center": {"lat": 10.55, "lon": 107.20},
        "zoom": 8.5,
    },
    "dong_nai": {
        "name": u("\\u0110\\u1ed3ng Nai"),
        "file": "pollution_index_scaled_dong_nai_2025.csv",
        "center": {"lat": 10.95, "lon": 107.05},
        "zoom": 8.0,
    },
    "ho_chi_minh": {
        "name": u("TP H\\u1ed3 Ch\\u00ed Minh"),
        "file": "pollution_index_scaled_ho_chi_minh_2025.csv",
        "center": {"lat": 10.78, "lon": 106.70},
        "zoom": 9.0,
    },
    "long_an": {
        "name": "Long An",
        "file": "pollution_index_scaled_long_an_2025.csv",
        "center": {"lat": 10.75, "lon": 106.10},
        "zoom": 8.0,
    },
    "tay_ninh": {
        "name": u("T\\u00e2y Ninh"),
        "file": "pollution_index_scaled_tay_ninh_2025.csv",
        "center": {"lat": 11.30, "lon": 106.15},
        "zoom": 8.5,
    },
}

COLOR_SCALE = [
    [0, "#66cc66"],
    [0.5, "#ffe066"],
    [1, "#ff6666"],
]

print("Project root:", ROOT_DIR)
print("Map mode:", MAP_MODE)
if MAP_MODE == "hourly":
    print("Target time:", TARGET_DATE, TARGET_HOUR)

## 1. Normalize names

GADM stores many Vietnamese names without spaces, for example `HoChiMinh`, `LongAn`, or `Quan1`. The pollution data uses regular district names. These helpers create matching keys for both sides.

In [ ]:
def strip_accents(value):
    value = str(value).replace(u("\\u0110"), "D").replace(u("\\u0111"), "d")
    normalized = unicodedata.normalize("NFKD", value)
    return normalized.encode("ascii", "ignore").decode("ascii")


def compact_key(value):
    return re.sub(r"[^a-z0-9]+", "", strip_accents(value).lower())


def province_key(value):
    aliases = {
        "bariavungtau": "ba_ria_vung_tau",
        "dongnai": "dong_nai",
        "hochiminh": "ho_chi_minh",
        "hochiminhcity": "ho_chi_minh",
        "tphochiminh": "ho_chi_minh",
        "longan": "long_an",
        "tayninh": "tay_ninh",
    }
    return aliases.get(compact_key(value), compact_key(value))


def district_key(value, province=None):
    text = strip_accents(value)
    text = re.sub(r"([a-z])([A-Z])", r"\1_\2", text)
    text = re.sub(r"([A-Za-z])(\d)", r"\1_\2", text)
    text = re.sub(r"(\d)([A-Za-z])", r"\1_\2", text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")

    if not re.fullmatch(r"quan_\d+", text):
        text = re.sub(r"^(tp|tx|huyen|quan|thanh_pho|thi_xa|district)_", "", text)
        text = re.sub(r"_(district|city|town)$", "", text)

    text = re.sub(r"_(tn|dn|la)$", "", text)

    district_aliases = {
        ("ba_ria_vung_tau", "tan_thanh"): "phu_my",
    }
    return district_aliases.get((province, text), text)


def make_merge_key(province, district):
    return f"{province}__{district}"

## 2. Load pollution index data

For `MAP_MODE = "hourly"`, the notebook filters one selected hour. For `MAP_MODE = "average"`, it uses the yearly average by district.

In [ ]:
def load_pollution_data():
    frames = []

    for p_key, config in PROVINCES.items():
        file_path = POLLUTION_DIR / config["file"]
        df = pd.read_csv(file_path, encoding="utf-8-sig")
        df["province_key"] = p_key
        df["province_name"] = config["name"]
        df["district_key"] = df["district"].apply(lambda value: district_key(value, p_key))
        df["merge_key"] = df.apply(lambda row: make_merge_key(row["province_key"], row["district_key"]), axis=1)
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


pollution = load_pollution_data()

if MAP_MODE == "hourly":
    selected = pollution[(pollution["date"] == TARGET_DATE) & (pollution["hour"] == TARGET_HOUR)].copy()
    if selected.empty:
        raise ValueError(f"No data found for {TARGET_DATE} {TARGET_HOUR}")
    map_title_time = f"{TARGET_DATE} {TARGET_HOUR}"
else:
    selected = pollution.copy()
    map_title_time = "2025 yearly average"

# Keep one row per district for mapping.
df_map = (
    selected.groupby(["province_key", "province_name", "district_key", "district", "merge_key"], as_index=False)
    .agg(
        lat=("lat", "mean"),
        lon=("lon", "mean"),
        pollution_index_scaled=("pollution_index_scaled", "mean"),
        pollution_index=("pollution_index", "mean"),
        records=("pollution_index_scaled", "size"),
    )
)

df_map["pollution_index_scaled"] = df_map["pollution_index_scaled"].round(2)
df_map["pollution_index"] = df_map["pollution_index"].round(0).astype(int)

print("Rows for map:", len(df_map))
display(df_map.head())
display(
    df_map.groupby("province_name")
    .agg(districts=("district", "count"), avg_scaled=("pollution_index_scaled", "mean"), max_scaled=("pollution_index_scaled", "max"))
    .round(2)
)

## 3. Load GeoJSON

The map uses GADM GeoJSON:

- Level 1 for Vietnam province context.
- Level 2 for district-level pollution polygons.

In [ ]:
def load_geojson(local_path, url):
    if local_path.exists():
        with open(local_path, "r", encoding="utf-8") as file:
            return json.load(file)

    with urlopen(url, timeout=60) as response:
        return json.load(response)


admin1_geojson = load_geojson(GADM_ADMIN1_LOCAL_PATH, GADM_ADMIN1_URL)
admin2_geojson = load_geojson(GADM_ADMIN2_LOCAL_PATH, GADM_ADMIN2_URL)

print("Admin level 1 features:", len(admin1_geojson["features"]))
print("Admin level 2 features:", len(admin2_geojson["features"]))

## 4. World / Vietnam context on tile map

This gives the wider context before focusing on the 5 target provinces.

In [ ]:
province_points = pd.DataFrame(
    [
        {
            "province_key": p_key,
            "province_name": config["name"],
            "lat": config["center"]["lat"],
            "lon": config["center"]["lon"],
        }
        for p_key, config in PROVINCES.items()
    ]
)

world_fig = px.scatter_map(
    province_points,
    lat="lat",
    lon="lon",
    hover_name="province_name",
    color_discrete_sequence=["#d73027"],
    zoom=3,
    center={"lat": 15.9, "lon": 106.5},
    map_style="open-street-map",
    title="World / Southeast Asia Context - Target Provinces in Vietnam",
    width=1000,
    height=650,
)
world_fig.update_traces(marker={"size": 12})
world_fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))
world_fig.show()

In [ ]:
def prepare_admin1_geojson(admin1):
    features = []
    rows = []

    for feature in admin1["features"]:
        props = feature["properties"]
        p_key = province_key(props["NAME_1"])
        is_target = p_key in PROVINCES

        feature = dict(feature)
        feature["properties"] = dict(props)
        feature["properties"].update({"province_key": p_key})
        features.append(feature)

        rows.append(
            {
                "province_key": p_key,
                "province_name": PROVINCES[p_key]["name"] if is_target else props["NAME_1"],
                "region": "Target provinces" if is_target else "Other Vietnam provinces",
                "highlight": 1 if is_target else 0,
            }
        )

    return {"type": "FeatureCollection", "features": features}, pd.DataFrame(rows)


admin1_map_geojson, vietnam_map_df = prepare_admin1_geojson(admin1_geojson)

vietnam_fig = px.choropleth_map(
    vietnam_map_df,
    geojson=admin1_map_geojson,
    locations="province_key",
    featureidkey="properties.province_key",
    color="region",
    color_discrete_map={
        "Target provinces": "#ff6666",
        "Other Vietnam provinces": "#d9d9d9",
    },
    hover_name="province_name",
    hover_data={"province_key": False, "highlight": False, "region": True},
    center={"lat": 16.1, "lon": 106.3},
    zoom=4.6,
    map_style="open-street-map",
    title="Vietnam Context - 5 Target Provinces",
    width=1000,
    height=850,
)
vietnam_fig.update_traces(marker_line_width=0.8, marker_line_color="#404040")
vietnam_fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))
vietnam_fig.show()

## 5. Prepare district GeoJSON and check matching

The district polygons are filtered to the 5 provinces, then merged with `df_map`.

In [ ]:
def prepare_admin2_geojson(admin2):
    features = []
    rows = []

    for feature in admin2["features"]:
        props = feature["properties"]
        p_key = province_key(props["NAME_1"])

        if p_key not in PROVINCES:
            continue

        d_key = district_key(props["NAME_2"], p_key)
        m_key = make_merge_key(p_key, d_key)

        feature = dict(feature)
        feature["properties"] = dict(props)
        feature["properties"].update(
            {
                "province_key": p_key,
                "district_key": d_key,
                "merge_key": m_key,
            }
        )
        features.append(feature)
        rows.append(
            {
                "province_key": p_key,
                "province_name": PROVINCES[p_key]["name"],
                "gadm_district": props["NAME_2"],
                "district_key": d_key,
                "merge_key": m_key,
            }
        )

    return {"type": "FeatureCollection", "features": features}, pd.DataFrame(rows)


district_geojson, geo_lookup = prepare_admin2_geojson(admin2_geojson)

matched = df_map.merge(
    geo_lookup[["merge_key", "gadm_district"]],
    on="merge_key",
    how="left",
    indicator=True,
)

unmatched_pollution = matched.loc[
    matched["_merge"] == "left_only",
    ["province_name", "district", "merge_key"],
]

unmatched_geojson = geo_lookup.merge(
    df_map[["merge_key"]],
    on="merge_key",
    how="left",
    indicator=True,
).loc[
    lambda data: data["_merge"] == "left_only",
    ["province_name", "gadm_district", "merge_key"],
]

matched = matched[matched["_merge"] == "both"].drop(columns="_merge")

print("Matched districts:", len(matched))
print("Unmatched pollution districts:", len(unmatched_pollution))
print("Unmatched GeoJSON districts:", len(unmatched_geojson))

display(unmatched_pollution)
display(unmatched_geojson)

## 6. Main pollution map, same style as the reference notebook

This is the main geographic distribution map: GeoJSON district polygons on OpenStreetMap tiles.

In [ ]:
def pollution_choropleth_map(data, geojson, title, center, zoom, width=1000, height=900):
    fig = px.choropleth_map(
        data,
        geojson=geojson,
        locations="merge_key",
        featureidkey="properties.merge_key",
        color="pollution_index_scaled",
        color_continuous_scale=COLOR_SCALE,
        range_color=[0, 100],
        hover_name="district",
        hover_data={
            "province_name": True,
            "pollution_index_scaled": ":.2f",
            "pollution_index": ":,",
            "records": ":,",
            "merge_key": False,
            "lat": False,
            "lon": False,
        },
        center=center,
        zoom=zoom,
        map_style="open-street-map",
        title=title,
        width=width,
        height=height,
    )

    fig.update_traces(marker_line_width=1.2, marker_line_color="#303030")
    fig.update_layout(
        margin=dict(l=0, r=0, t=50, b=0),
        coloraxis_colorbar={"title": "Pollution index"},
    )
    return fig


main_fig = pollution_choropleth_map(
    matched,
    district_geojson,
    f"Pollution Index - {map_title_time} - 5 Southeast Provinces",
    center={"lat": 10.85, "lon": 106.72},
    zoom=7.25,
    width=1050,
    height=900,
)
main_fig.show()

## 7. Separate map for each province

These maps use the same style but zoom into each province.

In [ ]:
if SHOW_PROVINCE_MAPS:
    for p_key, config in PROVINCES.items():
        province_data = matched[matched["province_key"] == p_key]
        province_geojson = {
            "type": "FeatureCollection",
            "features": [
                feature
                for feature in district_geojson["features"]
                if feature["properties"]["province_key"] == p_key
            ],
        }

        fig = pollution_choropleth_map(
            province_data,
            province_geojson,
            f"Pollution Index - {map_title_time} - {config['name']}",
            center=config["center"],
            zoom=config["zoom"],
            width=1000,
            height=800,
        )
        fig.show()

## Notes

- This notebook intentionally uses `px.choropleth_map`, matching the reference notebook style.
- `MAP_MODE = "hourly"` shows one selected hour; change `TARGET_DATE` and `TARGET_HOUR` to inspect another hour.
- `MAP_MODE = "average"` shows the yearly average by district.
- `Con Dao` appears in the pollution data, but this GADM admin level 2 GeoJSON does not include a matching polygon, so it appears in the unmatched table.
- GADM still uses `Tan Thanh`; the notebook maps it to `Phu My` so the data joins correctly.